In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [13]:
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

--2026-10-12 18:52:13--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv.3’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.003s  

2026-10-12 18:52:13 (76.0 MB/s) - ‘course_lead_scoring_2026.csv.3’ saved [278746/278746]



In [14]:
df = pd.read_csv("course_lead_scoring_2026.csv")
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [15]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [16]:
df = df.replace({
    "lead_source": {"NA": np.nan},
    "industry": {"NA": np.nan},
    "employment_status": {"NA": np.nan},
    "location": {"NA": np.nan},
    "annual_income": {0.0: np.nan, "0.0": np.nan},
    "lead_score": {0.0: np.nan, "0.0": np.nan}
})

In [17]:
df["industry"].mode()[0]

'technology'

In [18]:
numerical_features = ['number_of_courses_viewed', 'annual_income', 'interaction_count', 'lead_score']
df_numeric = df[numerical_features]

In [19]:
correlation_matrix = df_numeric.corr()
print("Correlation Matrix:")
print(correlation_matrix)

Correlation Matrix:
                          number_of_courses_viewed  annual_income  \
number_of_courses_viewed                  1.000000       0.137542   
annual_income                             0.137542       1.000000   
interaction_count                         0.721609       0.106375   
lead_score                                0.769352       0.231396   

                          interaction_count  lead_score  
number_of_courses_viewed           0.721609    0.769352  
annual_income                      0.106375    0.231396  
interaction_count                  1.000000    0.928712  
lead_score                         0.928712    1.000000  


In [20]:
print("\nSpecific Correlation Values (Absolute):")
print(f"interaction_count & lead_score: {abs(df['interaction_count'].corr(df['lead_score'])):.4f}")
print(f"number_of_courses_viewed & lead_score: {abs(df['number_of_courses_viewed'].corr(df['lead_score'])):.4f}")
print(f"number_of_courses_viewed & interaction_count: {abs(df['number_of_courses_viewed'].corr(df['interaction_count'])):.4f}")
print(f"annual_income & interaction_count: {abs(df['annual_income'].corr(df['interaction_count'])):.4f}")


Specific Correlation Values (Absolute):
interaction_count & lead_score: 0.9287
number_of_courses_viewed & lead_score: 0.7694
number_of_courses_viewed & interaction_count: 0.7216
annual_income & interaction_count: 0.1064


In [22]:
target = 'converted'

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

del df_train[target]
del df_val[target]
del df_test[target]


In [26]:
from sklearn.metrics import mutual_info_score

categorical = ['industry', 'location', 'lead_source', 'employment_status']

for col in categorical:

    feature_series = df_train[col].fillna('missing')
    
    mi = mutual_info_score(feature_series, y_train)
    print(f"{col}: {round(mi, 2)}")

industry: 0.0
location: 0.0
lead_source: 0.03
employment_status: 0.02


In [28]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

features = categorical + numerical_features

df_train_clean = df_train[features].copy()
df_val_clean = df_val[features].copy()

df_train_clean[categorical] = df_train_clean[categorical].fillna('missing')
df_val_clean[categorical] = df_val_clean[categorical].fillna('missing')

df_train_clean[numerical_features] = df_train_clean[numerical_features].fillna(0)
df_val_clean[numerical_features] = df_val_clean[numerical_features].fillna(0)

dv = DictVectorizer(sparse=False)
train_dicts = df_train_clean.to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val_clean.to_dict(orient='records')
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
baseline_accuracy = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy: {round(baseline_accuracy, 2)}")


Validation Accuracy: 0.65


In [29]:
baseline_accuracy = accuracy_score(y_val, model.predict(X_val))

for feature in features:
    subset_features = [f for f in features if f != feature]
    
    dv_sub = DictVectorizer(sparse=False)
    X_train_sub = dv_sub.fit_transform(df_train_clean[subset_features].to_dict(orient='records'))
    X_val_sub = dv_sub.transform(df_val_clean[subset_features].to_dict(orient='records'))
    
    model_sub = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_sub.fit(X_train_sub, y_train)
    accuracy_sub = accuracy_score(y_val, model_sub.predict(X_val_sub))
    
    print(f"Dropped: {feature:25} | Diff: {(baseline_accuracy - accuracy_sub):.4f}")


Dropped: industry                  | Diff: 0.0000
Dropped: location                  | Diff: 0.0000
Dropped: lead_source               | Diff: 0.0030
Dropped: employment_status         | Diff: 0.0000
Dropped: number_of_courses_viewed  | Diff: 0.0020
Dropped: annual_income             | Diff: -0.0790
Dropped: interaction_count         | Diff: 0.0440
Dropped: lead_score                | Diff: 0.0010


In [30]:
c_values = [0.000001, 0.00001, 0.0001, 0.001]

for c in c_values:
    model_reg = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model_reg.fit(X_train, y_train)
    
    accuracy_reg = accuracy_score(y_val, model_reg.predict(X_val))
    print(f"C = {c:<8} | Validation Accuracy: {round(accuracy_reg, 3)}")


C = 1e-06    | Validation Accuracy: 0.598
C = 1e-05    | Validation Accuracy: 0.598
C = 0.0001   | Validation Accuracy: 0.613
C = 0.001    | Validation Accuracy: 0.645
